In [1]:
import json
from pathlib import Path

import pandas as pd

In [2]:
pd.options.display.precision = 1

In [3]:
records = []

comparison_dir = Path("/data/connor/fmri-fm-eval/experiments/flops_benchmark")
paths = sorted((comparison_dir / "slurms").glob("slurm-*.out"))
print(len(paths))

paths.extend(sorted(Path("slurms").glob("slurm-*.out")))
print(len(paths))

for p in paths:
    with p.open() as f:
        for line in f:
            if line.startswith('{"model": '):
                records.append(json.loads(line))
df = pd.DataFrame.from_records(records)

# drop other flat mae model
df = df.query("model != 'flat_mae_base_patch16_2'")

df = df.sort_values(["model", "dataset"]).reset_index(drop=True)
df["params"] = df["params"] / 1e6

print(df.shape)

28
46
(65, 11)


In [4]:
df.head()

,model,dataset,input_shape,params,gflop,load_sps,load_fps,load_mbs,fwd_sps,fwd_fps,fwd_tflops
0,brain_harmonix_f,aabc_sex,"[500, 400]",89.4,3134.7,685.6,342783.1,548.5,39.1,19525.2,122.4
1,brain_harmonix_f,aabc_sex,"[500, 400]",89.4,3134.7,619.7,309827.1,495.7,39.1,19539.0,122.5
2,brain_harmonix_f,aabc_sex,"[500, 400]",89.4,3134.7,528.3,264138.9,422.6,39.0,19487.1,122.2
3,brain_harmonix_f,hcpya_task21,"[16, 400]",89.4,3134.5,2499.4,39990.9,64.0,39.0,624.7,122.4
4,brain_harmonix_f,hcpya_task21,"[16, 400]",89.4,3134.5,2169.4,34710.0,55.5,39.1,624.8,122.4


In [5]:
summary = df.groupby(["dataset", "model"]).agg(
    {
        "params": "median",
        "gflop": "median",
        "load_fps": "median",
        "fwd_fps": "median",
        "fwd_tflops": "median",
    }
)
summary

params   gflop  load_fps  fwd_fps  \
dataset      model                                                      
aabc_sex     brain_harmonix_f         89.4  3134.7  309827.1  2.0e+04   
             brain_jepa_vitb_ep300    86.8  1511.0  337463.2  8.6e+04   
             brain_semantoks          62.8     5.7  179159.7  1.1e+06   
             brainlm_vitmae_111m     112.6   382.3  267070.9  9.6e+04   
             flat_mae                 86.2  7234.5    3780.6  2.0e+04   
             mni_cortex_mae           87.0  9892.3    1457.2  1.5e+04   
             neurostorm                7.7   141.3     432.6  2.4e+04   
             schaefer400_mae          85.4  8062.1  379463.9  1.9e+04   
             swift                     4.3   182.5     455.5  1.5e+04   
hcpya_task21 brain_harmonix_f         89.4  3134.5   40427.8  6.3e+02   
             brain_jepa_vitb_ep300    86.8  1511.0   62295.2  2.8e+03   
             brain_semantoks          62.8     5.7   55932.3  3.8e+04   
             brainlm_vitmae_111m     112.6   191.1   30403.8  5.9e+03   
             flat_mae                 86.2   328.8    3749.3  1.4e+04   
             mni_cortex_mae           87.0   449.6    2071.8  1.0e+04   
             neurostorm                7.7    17.7     484.4  6.2e+03   
             schaefer400_mae          85.4   366.5   78918.5  1.3e+04   
             swift                     4.3    22.8     513.6  4.0e+03   

                                    fwd_tflops  
dataset      model                              
aabc_sex     brain_harmonix_f            122.4  
             brain_jepa_vitb_ep300       261.1  
             brain_semantoks              12.2  
             brainlm_vitmae_111m          73.0  
             flat_mae                    291.7  
             mni_cortex_mae              292.5  
             neurostorm                    6.7  
             schaefer400_mae             303.1  
             swift                         5.5  
hcpya_task21 brain_harmonix_f            122.5  
             brain_jepa_vitb_ep300       260.4  
             brain_semantoks              13.6  
             brainlm_vitmae_111m          71.0  
             flat_mae                    283.8  
             mni_cortex_mae              284.4  
             neurostorm                    6.9  
             schaefer400_mae             294.4  
             swift                         5.7

In [6]:
MODEL_NAMES = {
    "connectome_schaefer400": "Connectome",
    "identity_schaefer400": "MLP",
    "swift": "SwiFT",
    "brainlm_vitmae_111m": "BrainLM",
    "brain_jepa_vitb_ep300": "Brain-JEPA",
    "brain_harmonix_f": "BrainHarmonix-F",
    "neurostorm": "NeuroSTORM",
    "brain_semantoks": "Brain-Semantoks",
    "flat_mae": "CortexMAE-F",
    "schaefer400_mae": "CortexMAE-P",
    "mni_cortex_mae": "CortexMAE-V",
}

In [8]:
dataset = "hcpya_task21"

models = [
    "brainlm_vitmae_111m",
    "brain_jepa_vitb_ep300",
    "brain_harmonix_f",
    "brain_semantoks",
    "swift",
    "neurostorm",
    "schaefer400_mae",
    "flat_mae",
    "mni_cortex_mae",
]

records = []
for model in models:
    row = summary.loc[(dataset, model)]
    record = {
        "model": MODEL_NAMES[model],
        "params": f"{row['params']:.0f}M",
        "FLOP": f"{row['gflop']:.0f}G",
        "data": f"{round(row['load_fps'] / 1000, 1):.3g}K fps",
        "compute": f"{round(row['fwd_fps'] / 1000, 0):.4g}K fps",
        "FLOP/s": f"{round(row['fwd_tflops'], 1):.3g}T",
    }
    records.append(record)

summary_fmt = pd.DataFrame.from_records(records)
print(summary_fmt.to_markdown(index=False))
print(summary_fmt.to_latex(index=False))

| model           | params   | FLOP   | data      | compute   | FLOP/s   |
|:----------------|:---------|:-------|:----------|:----------|:---------|
| BrainLM         | 113M     | 191G   | 30.4K fps | 6K fps    | 71T      |
| Brain-JEPA      | 87M      | 1511G  | 62.3K fps | 3K fps    | 260T     |
| BrainHarmonix-F | 89M      | 3135G  | 40.4K fps | 1K fps    | 122T     |
| Brain-Semantoks | 63M      | 6G     | 55.9K fps | 38K fps   | 13.6T    |
| SwiFT           | 4M       | 23G    | 0.5K fps  | 4K fps    | 5.7T     |
| NeuroSTORM      | 8M       | 18G    | 0.5K fps  | 6K fps    | 6.9T     |
| CortexMAE-P     | 85M      | 366G   | 78.9K fps | 13K fps   | 294T     |
| CortexMAE-F     | 86M      | 329G   | 3.7K fps  | 14K fps   | 284T     |
| CortexMAE-V     | 87M      | 450G   | 2.1K fps  | 10K fps   | 284T     |
\begin{tabular}{llllll}
\toprule
model & params & FLOP & data & compute & FLOP/s \\
\midrule
BrainLM & 113M & 191G & 30.4K fps & 6K fps & 71T \\
Brain-JEPA & 87M & 1511G & 62.

aabc:

| model           | params   | FLOP   | data     | compute   | FLOP/s   |
|:----------------|:---------|:-------|:---------|:----------|:---------|
| BrainLM         | 113M     | 382G   | 267K fps | 96K fps   | 73T      |
| Brain-JEPA      | 87M      | 1511G  | 338K fps | 86K fps   | 261T     |
| BrainHarmonix-F | 89M      | 3135G  | 310K fps | 20K fps   | 122T     |
| Brain-Semantoks | 63M      | 6G     | 179K fps | 1079K fps | 12.2T    |
| SwiFT           | 4M       | 183G   | 0.5K fps | 15K fps   | 5.5T     |
| NeuroSTORM      | 8M       | 141G   | 0.4K fps | 24K fps   | 6.7T     |
| CortexMAE-P     | 85M      | 8062G  | 380K fps | 19K fps   | 303T     |
| CortexMAE-F     | 86M      | 7234G  | 3.8K fps | 20K fps   | 292T     |
| CortexMAE-V     | 87M      | 9892G  | 1.5K fps | 15K fps   | 292T     |

hcpya:

| model           | params   | FLOP   | data      | compute   | FLOP/s   |
|:----------------|:---------|:-------|:----------|:----------|:---------|
| BrainLM         | 113M     | 191G   | 30.4K fps | 6K fps    | 71T      |
| Brain-JEPA      | 87M      | 1511G  | 62.3K fps | 3K fps    | 260T     |
| BrainHarmonix-F | 89M      | 3135G  | 40.4K fps | 1K fps    | 122T     |
| Brain-Semantoks | 63M      | 6G     | 55.9K fps | 38K fps   | 13.6T    |
| SwiFT           | 4M       | 23G    | 0.5K fps  | 4K fps    | 5.7T     |
| NeuroSTORM      | 8M       | 18G    | 0.5K fps  | 6K fps    | 6.9T     |
| CortexMAE-P     | 85M      | 366G   | 78.9K fps | 13K fps   | 294T     |
| CortexMAE-F     | 86M      | 329G   | 3.7K fps  | 14K fps   | 284T     |
| CortexMAE-V     | 87M      | 450G   | 2.1K fps  | 10K fps   | 284T     |